# 1D exact full-likelihood Matérn: sample-size scaling

This notebook avoids Vecchia completely.

For each sample size `n = 200, 400, 800, 1600, 3200`, it simulates a fresh
1D Gaussian process from an exact dense Matérn covariance, then fits by dense
Cholesky exact Gaussian likelihood.

The key diagnostic is whether smooth misspecification alone creates systematic
drift in `sigmasq`, `range`, and `nugget` as `n` increases.


In [ ]:
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from scipy.special import gamma as scipy_gamma, kv as scipy_kv
from scipy.linalg import cho_factor, cho_solve

warnings.filterwarnings('ignore', category=RuntimeWarning)

ROUND_DECIMALS = 4
PURE_SPACE_DIR = Path('/Users/joonwonlee/Documents/GEMS_TCO-1/Exercises/st_model/day/local_computer/pure_space')
OUT_DIR = PURE_SPACE_DIR / 'log'
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PREFIX = 'sim_1d_exact_full_likelihood_nscale_r0p2_050826'

def round_df(df, ndigits=ROUND_DECIMALS):
    out = df.copy()
    for c in out.select_dtypes(include=[np.number]).columns:
        out[c] = out[c].round(ndigits)
    return out


In [ ]:
# Experiment config.
TRUE_SMOOTHS = [0.3, 0.5, 1.5]
FIT_SMOOTHS = [0.3, 0.5, 1.5]
N_LIST = [200, 400, 800, 1600, 3200]
N_REPS = 1
SIM_SEED = 20260508

TRUE_PARAMS = {
    'sigmasq': 12.0,
    'range': 0.20,
    'nugget': 2.0,
}

FIT_TYPES = {
    'full_nugget_free': {
        'free': ['sigmasq', 'range', 'nugget'],
        'fixed': {},
    },
    'full_nugget0': {
        'free': ['sigmasq', 'range'],
        'fixed': {'nugget': 0.0},
    },
    'sigma_only_nugget_free': {
        'free': ['sigmasq'],
        'fixed': {'range': TRUE_PARAMS['range'], 'nugget': TRUE_PARAMS['nugget']},
    },
    'range_only_nugget_free': {
        'free': ['range'],
        'fixed': {'sigmasq': TRUE_PARAMS['sigmasq'], 'nugget': TRUE_PARAMS['nugget']},
    },
    'nugget_only': {
        'free': ['nugget'],
        'fixed': {'sigmasq': TRUE_PARAMS['sigmasq'], 'range': TRUE_PARAMS['range']},
    },
    'sigma_only_nugget0': {
        'free': ['sigmasq'],
        'fixed': {'range': TRUE_PARAMS['range'], 'nugget': 0.0},
    },
    'range_only_nugget0': {
        'free': ['range'],
        'fixed': {'sigmasq': TRUE_PARAMS['sigmasq'], 'nugget': 0.0},
    },
}

# Dense Cholesky at n=3200 is exact but not cheap. Start with the core set,
# then expand to list(FIT_TYPES) if needed.
FIT_TYPES_TO_RUN = [
    'full_nugget_free',
    'full_nugget0',
    'sigma_only_nugget_free',
    'range_only_nugget_free',
    'nugget_only',
    'sigma_only_nugget0',
    'range_only_nugget0',
]

PARAM_BOUNDS = {
    'sigmasq': (1e-4, 300.0),
    'range': (0.003, 1.0),
    'nugget': (1e-8, 150.0),
}
INIT_PARAMS = {'sigmasq': 10.0, 'range': 0.20, 'nugget': 1.0}
JITTER = 1e-7

# Fewer iterations for large n; warm starts carry estimates forward.
MAXITER_BY_N = {200: 90, 400: 75, 800: 55, 1600: 35, 3200: 25}

print('true smooths:', TRUE_SMOOTHS)
print('fit smooths:', FIT_SMOOTHS)
print('n list:', N_LIST)
print('fit types:', FIT_TYPES_TO_RUN)


In [ ]:
def matern_corr_1d(dist_over_range, smooth):
    r = np.asarray(dist_over_range, dtype=np.float64)
    nu = float(smooth)
    if np.isclose(nu, 0.5):
        return np.exp(-r)
    z = np.sqrt(2.0 * nu) * r
    safe = np.where(z < 1e-12, 1e-12, z)
    out = (2.0 ** (1.0 - nu) / scipy_gamma(nu)) * (safe ** nu) * scipy_kv(nu, safe)
    out = np.where(z < 1e-12, 1.0, out)
    return np.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0)


def cov_1d_from_dist(dist, smooth, sigmasq, range_, nugget=0.0):
    K = float(sigmasq) * matern_corr_1d(dist / float(range_), smooth)
    if float(nugget) > 0:
        K = K + float(nugget) * np.eye(dist.shape[0])
    return K


def make_distance_matrix(n):
    x = np.linspace(0.0, 1.0, int(n))
    return x, np.abs(x[:, None] - x[None, :])


def simulate_exact_1d(rng, dist, smooth, params):
    K = cov_1d_from_dist(dist, smooth, params['sigmasq'], params['range'], params['nugget'])
    K = K + JITTER * np.eye(dist.shape[0])
    L = np.linalg.cholesky(K)
    return L @ rng.standard_normal(dist.shape[0])


def neg_loglik_from_dist(y, dist, smooth, sigmasq, range_, nugget):
    n = len(y)
    K = cov_1d_from_dist(dist, smooth, sigmasq, range_, nugget) + JITTER * np.eye(n)
    try:
        cf = cho_factor(K, lower=True, check_finite=False)
        alpha = cho_solve(cf, y, check_finite=False)
        logdet = 2.0 * np.sum(np.log(np.diag(cf[0])))
        return 0.5 * (logdet + float(y @ alpha) + n * np.log(2.0 * np.pi))
    except Exception:
        return np.inf


def unpack_params(theta, free_names, fixed):
    vals = dict(fixed)
    for name, logv in zip(free_names, theta):
        vals[name] = float(np.exp(logv))
    for name in ['sigmasq', 'range', 'nugget']:
        vals.setdefault(name, TRUE_PARAMS[name])
    return vals


def fit_exact_1d(y, dist, fit_smooth, fit_type, init_params=None, maxiter=60):
    spec = FIT_TYPES[fit_type]
    free = list(spec['free'])
    fixed = dict(spec['fixed'])
    init = dict(INIT_PARAMS)
    if init_params:
        init.update({k: float(v) for k, v in init_params.items() if k in init})
    theta0 = np.array([np.log(np.clip(init[name], *PARAM_BOUNDS[name])) for name in free], dtype=float)
    bounds = [(np.log(PARAM_BOUNDS[name][0]), np.log(PARAM_BOUNDS[name][1])) for name in free]

    def obj(theta):
        p = unpack_params(theta, free, fixed)
        return neg_loglik_from_dist(y, dist, fit_smooth, p['sigmasq'], p['range'], p['nugget'])

    if free:
        opt = minimize(
            obj,
            theta0,
            method='L-BFGS-B',
            bounds=bounds,
            options={'maxiter': int(maxiter), 'ftol': 1e-8, 'maxls': 25},
        )
        theta_hat = opt.x
        loss = float(opt.fun)
        success = bool(opt.success)
        message = str(opt.message)
        n_iter = int(getattr(opt, 'nit', -1))
        n_eval = int(getattr(opt, 'nfev', -1))
    else:
        theta_hat = np.array([])
        loss = float(obj(theta_hat))
        success = True
        message = 'fixed'
        n_iter = 0
        n_eval = 1
    est = unpack_params(theta_hat, free, fixed)
    return est, loss / len(y), success, n_iter, n_eval, message


In [ ]:
# Simulate fresh data at each n. This is not thinning from a bigger sample.
rng = np.random.default_rng(SIM_SEED)
dist_cache = {}
sim_data = {}
sim_rows = []

for n in N_LIST:
    x, dist = make_distance_matrix(n)
    dist_cache[int(n)] = dist
    for true_smooth in TRUE_SMOOTHS:
        for rep in range(N_REPS):
            y = simulate_exact_1d(rng, dist, true_smooth, TRUE_PARAMS)
            sim_data[(int(n), float(true_smooth), int(rep))] = y
            sim_rows.append({
                'n': int(n),
                'true_smooth': float(true_smooth),
                'rep': int(rep),
                'y_mean': float(np.mean(y)),
                'y_var': float(np.var(y, ddof=1)),
            })

sim_df = pd.DataFrame(sim_rows)
sim_path = OUT_DIR / f'{OUT_PREFIX}_sim_summary.csv'
round_df(sim_df).to_csv(sim_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved simulation summary:', sim_path)
display(round_df(sim_df))


In [ ]:
# Exact full likelihood fits.
results = []
warm = {}
t_all = time.time()

for true_smooth in TRUE_SMOOTHS:
    for fit_smooth in FIT_SMOOTHS:
        for fit_type in FIT_TYPES_TO_RUN:
            for rep in range(N_REPS):
                last_est = None
                for n in N_LIST:
                    y = sim_data[(int(n), float(true_smooth), int(rep))]
                    dist = dist_cache[int(n)]
                    maxiter = MAXITER_BY_N.get(int(n), 50)
                    t0 = time.time()
                    est, loss_per_obs, success, n_iter, n_eval, message = fit_exact_1d(
                        y,
                        dist,
                        fit_smooth=fit_smooth,
                        fit_type=fit_type,
                        init_params=last_est,
                        maxiter=maxiter,
                    )
                    elapsed = time.time() - t0
                    last_est = est
                    row = {
                        'n': int(n),
                        'true_smooth': float(true_smooth),
                        'fit_smooth': float(fit_smooth),
                        'rep': int(rep),
                        'fit_type': fit_type,
                        'loss_per_obs': float(loss_per_obs),
                        'success': bool(success),
                        'n_iter': int(n_iter),
                        'n_eval': int(n_eval),
                        'fit_s': float(elapsed),
                        'message': message,
                        'est_sigmasq': float(est['sigmasq']),
                        'est_range': float(est['range']),
                        'est_nugget': float(est['nugget']),
                        'true_sigmasq': TRUE_PARAMS['sigmasq'],
                        'true_range': TRUE_PARAMS['range'],
                        'true_nugget': TRUE_PARAMS['nugget'],
                    }
                    results.append(row)
                    tmp = pd.DataFrame(results)
                    tmp_path = OUT_DIR / f'{OUT_PREFIX}_fits_partial.csv'
                    round_df(tmp).to_csv(tmp_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
                    print(
                        f"true={true_smooth} fit={fit_smooth} {fit_type} rep={rep} n={n} "
                        f"loss={loss_per_obs:.4f} range={est['range']:.4f} nugget={est['nugget']:.4f} "
                        f"time={elapsed:.1f}s success={success}"
                    )

fit_df = pd.DataFrame(results)
fit_path = OUT_DIR / f'{OUT_PREFIX}_fits.csv'
round_df(fit_df).to_csv(fit_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved fits:', fit_path)
print('Total elapsed:', time.time() - t_all)
display(round_df(fit_df.head(20)))


In [ ]:
# Summary.
summary_df = (
    fit_df
    .groupby(['true_smooth', 'fit_smooth', 'n', 'fit_type'], observed=False)
    .agg(
        n_reps=('rep', 'nunique'),
        loss_mean=('loss_per_obs', 'mean'),
        sigmasq_mean=('est_sigmasq', 'mean'),
        sigmasq_sd=('est_sigmasq', 'std'),
        range_mean=('est_range', 'mean'),
        range_sd=('est_range', 'std'),
        nugget_mean=('est_nugget', 'mean'),
        nugget_sd=('est_nugget', 'std'),
        success_rate=('success', 'mean'),
        fit_s_mean=('fit_s', 'mean'),
    )
    .reset_index()
)
summary_path = OUT_DIR / f'{OUT_PREFIX}_summary.csv'
round_df(summary_df).to_csv(summary_path, index=False, float_format=f'%.{ROUND_DECIMALS}f')
print('Saved summary:', summary_path)
display(round_df(summary_df.head(30)))


In [ ]:
# Plot helpers.
PARAM_INFO = {
    'sigmasq': ('est_sigmasq', TRUE_PARAMS['sigmasq']),
    'range': ('est_range', TRUE_PARAMS['range']),
    'nugget': ('est_nugget', TRUE_PARAMS['nugget']),
}

def plot_nscale_grid(fit_type, parameter, y_shared=True):
    y_col, true_value = PARAM_INFO[parameter]
    sub = fit_df[fit_df['fit_type'] == fit_type].copy()
    if sub.empty:
        print('No data for', fit_type)
        return
    stats = (
        sub
        .groupby(['true_smooth', 'fit_smooth', 'n'], observed=False)
        .agg(mean=(y_col, 'mean'), sd=(y_col, 'std'))
        .reset_index()
        .sort_values(['true_smooth', 'fit_smooth', 'n'])
    )
    fig, axes = plt.subplots(len(TRUE_SMOOTHS), len(FIT_SMOOTHS), figsize=(14, 10), constrained_layout=True, sharex=True, sharey=y_shared)
    for i, true_smooth in enumerate(TRUE_SMOOTHS):
        for j, fit_smooth in enumerate(FIT_SMOOTHS):
            ax = axes[i, j]
            s = stats[(stats['true_smooth'] == true_smooth) & (stats['fit_smooth'] == fit_smooth)]
            yerr = s['sd'].fillna(0.0) if 'sd' in s else None
            ax.errorbar(s['n'], s['mean'], yerr=yerr, marker='o', capsize=3)
            ax.axhline(true_value, color='black', linestyle=':', linewidth=1.2)
            ax.set_xscale('log', base=2)
            ax.set_xticks(N_LIST, [str(n) for n in N_LIST])
            ax.set_title(f'true nu={true_smooth}, fit nu={fit_smooth}')
            ax.grid(True, alpha=0.25)
            if j == 0:
                ax.set_ylabel(parameter)
            if i == len(TRUE_SMOOTHS) - 1:
                ax.set_xlabel('n')
    fig.suptitle(f'Exact full likelihood, {fit_type}, parameter={parameter}', fontsize=14)
    out = OUT_DIR / f'{OUT_PREFIX}_{fit_type}_{parameter}_nscale_grid.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('Saved:', out)
    plt.show()


def plot_loss_grid(fit_type, y_shared=True):
    sub = fit_df[fit_df['fit_type'] == fit_type].copy()
    stats = (
        sub
        .groupby(['true_smooth', 'fit_smooth', 'n'], observed=False)
        .agg(mean=('loss_per_obs', 'mean'), sd=('loss_per_obs', 'std'))
        .reset_index()
        .sort_values(['true_smooth', 'fit_smooth', 'n'])
    )
    fig, axes = plt.subplots(len(TRUE_SMOOTHS), len(FIT_SMOOTHS), figsize=(14, 10), constrained_layout=True, sharex=True, sharey=y_shared)
    for i, true_smooth in enumerate(TRUE_SMOOTHS):
        for j, fit_smooth in enumerate(FIT_SMOOTHS):
            ax = axes[i, j]
            s = stats[(stats['true_smooth'] == true_smooth) & (stats['fit_smooth'] == fit_smooth)]
            ax.errorbar(s['n'], s['mean'], yerr=s['sd'].fillna(0.0), marker='o', capsize=3)
            ax.set_xscale('log', base=2)
            ax.set_xticks(N_LIST, [str(n) for n in N_LIST])
            ax.set_title(f'true nu={true_smooth}, fit nu={fit_smooth}')
            ax.grid(True, alpha=0.25)
            if j == 0:
                ax.set_ylabel('negative loglik / obs')
            if i == len(TRUE_SMOOTHS) - 1:
                ax.set_xlabel('n')
    fig.suptitle(f'Exact full likelihood loss, {fit_type}', fontsize=14)
    out = OUT_DIR / f'{OUT_PREFIX}_{fit_type}_loss_nscale_grid.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('Saved:', out)
    plt.show()


In [ ]:
# Main plots.
for fit_type in FIT_TYPES_TO_RUN:
    for parameter in ['sigmasq', 'range', 'nugget']:
        if parameter == 'nugget' and FIT_TYPES[fit_type]['fixed'].get('nugget', None) == 0.0:
            continue
        if parameter not in ['sigmasq', 'range', 'nugget']:
            continue
        plot_nscale_grid(fit_type, parameter)
    plot_loss_grid(fit_type)


In [ ]:
# Heatmaps at selected n.
HEATMAP_N = 3200

def plot_heatmap(fit_type, parameter, n=HEATMAP_N):
    y_col, true_value = PARAM_INFO[parameter]
    sub = fit_df[(fit_df['fit_type'] == fit_type) & (fit_df['n'] == int(n))].copy()
    if sub.empty:
        print('No rows for', fit_type, n)
        return
    mat = (
        sub
        .groupby(['true_smooth', 'fit_smooth'], observed=False)[y_col]
        .mean()
        .unstack('fit_smooth')
        .reindex(index=TRUE_SMOOTHS, columns=FIT_SMOOTHS)
    )
    bias = mat - true_value
    fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
    im0 = axes[0].imshow(mat.to_numpy(), aspect='auto')
    im1 = axes[1].imshow(bias.to_numpy(), aspect='auto', cmap='coolwarm')
    axes[0].set_title(f'{parameter} estimate')
    axes[1].set_title(f'{parameter} bias')
    for ax, arr in [(axes[0], mat.to_numpy()), (axes[1], bias.to_numpy())]:
        ax.set_xticks(range(len(FIT_SMOOTHS)), FIT_SMOOTHS)
        ax.set_yticks(range(len(TRUE_SMOOTHS)), TRUE_SMOOTHS)
        ax.set_xlabel('fit smooth')
        ax.set_ylabel('true smooth')
        for i in range(arr.shape[0]):
            for j in range(arr.shape[1]):
                ax.text(j, i, f'{arr[i, j]:.3g}', ha='center', va='center', color='white')
    fig.colorbar(im0, ax=axes[0], shrink=0.8)
    fig.colorbar(im1, ax=axes[1], shrink=0.8)
    fig.suptitle(f'{fit_type}, n={n}', fontsize=13)
    out = OUT_DIR / f'{OUT_PREFIX}_{fit_type}_{parameter}_n{n}_heatmap.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('Saved:', out)
    plt.show()

for fit_type in ['full_nugget_free', 'full_nugget0', 'sigma_only_nugget_free', 'range_only_nugget_free', 'nugget_only']:
    if fit_type not in FIT_TYPES_TO_RUN:
        continue
    for parameter in ['sigmasq', 'range', 'nugget']:
        if parameter == 'nugget' and FIT_TYPES[fit_type]['fixed'].get('nugget', None) == 0.0:
            continue
        if parameter == 'range' and 'range' not in FIT_TYPES[fit_type]['free']:
            continue
        if parameter == 'sigmasq' and 'sigmasq' not in FIT_TYPES[fit_type]['free']:
            continue
        plot_heatmap(fit_type, parameter)


In [ ]:
# Optional quick compare table at n=3200.
compare_cols = [
    'true_smooth', 'fit_smooth', 'fit_type', 'n',
    'loss_mean', 'sigmasq_mean', 'range_mean', 'nugget_mean', 'success_rate'
]
display(
    round_df(
        summary_df[summary_df['n'] == max(N_LIST)][compare_cols]
        .sort_values(['fit_type', 'true_smooth', 'fit_smooth'])
    )
)
